# LLM Evaluation on Greek Protipa Exams

In [1]:
import json
import logging
import requests
import os
import sys
import random
import time
import traceback
from pathlib import Path

import lm_eval
from lm_eval.utils import make_table
import pandas as pd
import yaml
from datasets import load_dataset
from datasets import load_dataset, concatenate_datasets
from dotenv import load_dotenv, find_dotenv
from lm_eval.models.openai_completions import OpenAIChatCompletion
from lm_eval.tasks import ConfigurableTask, TaskManager

import IPython.display

# Setup Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

## 1. Environment Setup

In [6]:
load_dotenv(find_dotenv())

# API Config
os.environ["LITELLM_ILSP_EVAL_API_KEY"] = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
os.environ["LITELLM_HOST"] = os.getenv("LITELLM_HOST")
os.environ["OPENAI_API_KEY"] = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
api_base = os.getenv("LITELLM_HOST")

# Output Config
results_dir = Path(os.getenv("RESULTS_DIR", "tmp"))
results_dir.mkdir(parents=True, exist_ok=True)

models_to_test = ["gemma3-27b-it", "krikri-dpo-context"]
logger.info(f"Target models: {models_to_test}")

2026-05-06 16:13:12 - INFO - Target models: ['gemma3-27b-it', 'krikri-dpo-context']


In [ ]:
project_root = Path.cwd().parent 
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    logger.info(f"✅ Προστέθηκε το {src_path} στο path!")

try:
    from protipa_exams_dataset.data_loader import load_protipa_dataset, apply_matching_processing
    logger.info("🚀 Επιτυχία! Το data_loader φορτώθηκε από το src.")
except ImportError as e:
    logger.warning(f"⚠️ Δεν βρέθηκε η συνάρτηση/module. Έλεγξε τα ονόματα στο src. Error: {e}")

Διαθέσιμα μοντέλα

In [7]:
#api_key = os.getenv("OPENAI_API_KEY")
#api_base = os.getenv("OPENAI_BASE_URL")

api_key = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
api_base = os.getenv("LITELLM_HOST")

# Καθαρισμός URL
if api_base.endswith("/chat/completions"):
    api_base = api_base.replace("/chat/completions", "")
if not api_base.endswith("/v1"):
    api_base = api_base.rstrip("/") + "/v1"

try:
    response = requests.get(
        f"{api_base}/models", 
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=10
    )
    
    if response.status_code == 200:
        data = response.json()
        models = data.get('data', [])
        
        targets = ['llama', 'mistral', 'gemma', 'gpt']
        found_models = []
        
        for m in models:
            mid = m['id']
            if any(t in mid.lower() for t in targets):
                found_models.append(mid)
        
        print(json.dumps(found_models, indent=4))
        
    else:
        print(f"Error: {response.text}")

except Exception as e:
    print(f"Connection Error: {e}")

[
    "llama-krikri-8b-instruct-v1.5"
]


In [3]:
#Saving results from closed tasks

file_path = "../results/closed_aggregate_test/llama-krikri-8b-instruct-v1.5/closed_aggregate_results.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/closed_aggregate_test/llama-krikri-8b-instruct-v1.5/krikri_closed_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

Table was successfully saved in results folder!
|                     Tasks                     |Version|   Filter   |n-shot|  Metric   |   |Value |   |Stderr|
|-----------------------------------------------|-------|------------|-----:|-----------|---|-----:|---|-----:|
| - greek_protipa_exams_language_closed         |Yaml   |strict-match|     0|exact_match|↑  |0.4933|±  |0.0410|
| - greek_protipa_exams_maths_closed            |Yaml   |strict-match|     0|exact_match|↑  |0.2267|±  |0.0343|
| - greek_protipa_exams_religious_studies_closed|Yaml   |strict-match|     0|exact_match|↑  |0.7444|±  |0.0462|



In [2]:
#Saving results from closed tasks_private_few_shot

file_path = "../results/closed_aggregate_private/llama-krikri-8b-instruct-v1.5/closed_aggregate_private_results.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/closed_aggregate_private/llama-krikri-8b-instruct-v1.5/krikri_closed_private_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

Table was successfully saved in results folder!
|                         Tasks                         |Version|   Filter   |n-shot|  Metric   |   |Value|   |Stderr|
|-------------------------------------------------------|-------|------------|-----:|-----------|---|----:|---|-----:|
| - greek_protipa_exams_language_closed_private         |Yaml   |strict-match|     5|exact_match|↑  |  0.6|±  |0.0700|
| - greek_protipa_exams_maths_closed_private            |Yaml   |strict-match|     5|exact_match|↑  |  0.3|±  |0.0655|
| - greek_protipa_exams_religious_studies_closed_private|Yaml   |strict-match|     5|exact_match|↑  |  0.6|±  |0.0700|



In [4]:
#Saving results from open tasks

file_path = "../results/open_aggregate_test/llama-krikri-8b-instruct-v1.5/open_aggregate_results.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/open_aggregate_test/llama-krikri-8b-instruct-v1.5/krikri_open_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

Table was successfully saved in results folder!
|               Tasks                |Version|Filter|n-shot|     Metric     |   | Value |   |Stderr|
|------------------------------------|-------|------|-----:|----------------|---|------:|---|-----:|
|greek_protipa_exams_open_aggregate  |       |none  |      |bertscore_f1_max|↑  | 0.6854|±  |0.0035|
| - greek_protipa_exams_language_open|Yaml   |none  |     0|bertscore_f1_max|↑  | 0.6841|±  |0.0055|
|                                    |       |none  |     0|bleu_max        |   | 3.6652|±  |0.7410|
|                                    |       |none  |     0|rouge1_max      |   |18.5077|±  |1.3359|
|                                    |       |none  |     0|rouge2_max      |   | 7.0270|±  |1.0651|
|                                    |       |none  |     0|rougeL_max      |   |15.7558|±  |1.1707|
| - greek_protipa_exams_maths_open   |Yaml   |none  |     0|bertscore_f1_max|↑  | 0.6939|±  |0.0044|
|                                    |     

In [2]:
#Saving results from open tasks_private_few_shot

file_path = "../results/open_aggregate_private/llama-krikri-8b-instruct-v1.5/open_aggregate_private_results.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/open_aggregate_private/llama-krikri-8b-instruct-v1.5/krikri_open_private_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

Table was successfully saved in results folder!
|                   Tasks                    |Version|Filter|n-shot|     Metric     |   | Value |   |Stderr|
|--------------------------------------------|-------|------|-----:|----------------|---|------:|---|-----:|
|greek_protipa_exams_open_aggregate_private  |       |none  |      |bertscore_f1_max|↑  | 0.6810|±  |0.0074|
| - greek_protipa_exams_language_open_private|Yaml   |none  |     5|bertscore_f1_max|↑  | 0.6766|±  |0.0099|
|                                            |       |none  |     5|bleu_max        |   | 3.5715|±  |1.5060|
|                                            |       |none  |     5|rouge1_max      |   |14.8813|±  |2.8313|
|                                            |       |none  |     5|rouge2_max      |   | 5.3899|±  |1.9188|
|                                            |       |none  |     5|rougeL_max      |   |14.1045|±  |2.7553|
| - greek_protipa_exams_maths_open_private   |Yaml   |none  |     5|bertscore_f1

In [10]:
#Saving results from structured tasks

file_path = "../results/structured_aggregate_test/llama-krikri-8b-instruct-v1.5/structured_aggregate_results.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/structured_aggregate_test/llama-krikri-8b-instruct-v1.5/krikri_structured_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

Table was successfully saved in results folder!
|                  Tasks                   |Version|Filter|n-shot|      Metric       |   |Value |   |Stderr|
|------------------------------------------|-------|------|-----:|-------------------|---|-----:|---|-----:|
|greek_protipa_exams_structured_aggregate  |       |none  |      |structured_accuracy|↑  |0.2415|±  |0.0589|
| - greek_protipa_exams_language_structured|Yaml   |none  |     0|structured_accuracy|↑  |0.2415|±  |0.0589|



In [3]:
#Saving results from structured tasks_private_few_shot

file_path = "../results/structured_aggregate_private/llama-krikri-8b-instruct-v1.5/structured_aggregate_private_results.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/structured_aggregate_private/llama-krikri-8b-instruct-v1.5/krikri_structured_results_private_few_shot.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

Table was successfully saved in results folder!
|                      Tasks                       |Version|Filter|n-shot|      Metric       |   |Value |   |Stderr|
|--------------------------------------------------|-------|------|-----:|-------------------|---|-----:|---|-----:|
|greek_protipa_exams_structured_aggregate_private  |       |none  |      |structured_accuracy|↑  |0.2528|±  |0.0591|
| - greek_protipa_exams_language_structured_private|Yaml   |none  |     5|structured_accuracy|↑  |0.2528|±  |0.0591|

